### Innovation.
This new version improves upon the last iteration in the following domain:
- sensors precision and ODR from datasheets
- regular sensor sampling rate
- linear kalman filtering on height and its two derivatives

Further iterations shall incorporate a non-linear filter that includes a linearization for $v^2$, $C_d$ as well as $u$ and **B** for modelling the control outputs of the airbrakes. 

In [ ]:
import queue
import numpy as np
from typing import Callable


class HiddenSystemState: 
    acceleration:float = None
    velocity:float  = None
    height: float = None

    temperature: float = None
    air_density: float = None
    air_pressure: float = None

    def __init__(self,a,v,h,t,ad,ap):
        self.acceleration = a
        self.velocity = v
        self.height = h
        self.temperature = t
        self.air_density = ad
        self.air_pressure = ap

    
class Sensor:
    std_dev: float = None
    name: str = ""
    output_data_rate: int = 1 # in Hz
    output_data: queue.Queue = queue.Queue()

    systemState: HiddenSystemState = None
    lastTickTs: float = 0    

    def __init__(self,name_, std_dev_, odr_, systemStateRef_, getTargetVar_: Callable[[HiddenSystemState],float]):
        self.name = name_
        self.std_dev = std_dev_
        self.output_data_rate = odr_
        self.systemState = systemStateRef_
        self.getTargetValue = getTargetVar_

    def _generateSample(self) -> float:
        physicalValue:float = self.getTargetValue(self.systemState)
        z = np.random.normal(physicalValue,self.std_dev,1)
        return z

    def reset(self):
        self.lastTickTs = 0
        self.output_data.clear()

    def tick(self, ts:float):
        """
        Checks the time that has passed since last tick and populates the output_data queue
        according to the ODR.
        """
        if ts - self.lastTickTs > (1.0 / self.output_data_rate):
            self.lastTickTs = ts
            sample = self._generateSample()
            self.output_data.put(sample)

The IMU's have different operation modes, from 4 to 32 or 24g. The maximum deceleration we will experience after motor burnout is (without airbrakes) about -1.21G which means we are safe to use the finest resolution of 4G and will continue under this assumption. 

Important: Testing is required to check, whether the barometers precision holds under the rockets conditions, as it is in a closed tube and experiences acceleration.

In [31]:
# Setup States and Sensors
sysState = HiddenSystemState(-9.8, 129, 250,21,1.256)

primary_imu = Sensor("LSM6DSO32",0.03139,1600,sysState,lambda s: s.acceleration)
secondary_imu = Sensor("BMI088",0.0262644,1600,sysState, lambda s: s.acceleration)
barometer = Sensor("DPS368",0.35,26,sysState,lambda s: s.air_pressure)